# CUDA reciprocal approximation with TR-ln and Q8 TR-exp

Compare the CUDA reciprocal helpers from `trptq_softmax_dp4a` against floating-point `1 / x` baselines:

- `reciprocal_u8`: raw int32 denominator -> saturated uint8 reciprocal, scaled by `2**8`.
- `reciprocal_u16`: raw int32 denominator -> saturated uint16 reciprocal, scaled by `2**16`.
- `reciprocal_q16_i32`: existing legacy/debug int32 Q16 path.


In [ ]:
from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f"Drive mount skipped: {exc}")

In [ ]:
import math
import os
import sys
from pathlib import Path

import torch

PROJECT_DIR_PATH = globals().get("PROJECT_DIR_PATH", "/content/mrcp-tr-ptq")
PROJECT_DIR = Path(PROJECT_DIR_PATH).resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

%cd {PROJECT_DIR_PATH}

INT32_MAX = (1 << 31) - 1
RECIP_UINT8_SCALE = 1 << 8
RECIP_UINT16_SCALE = 1 << 16
RECIP_Q16_SCALE = 1 << 16
SOFTMAX_EXT_DIR = PROJECT_DIR / "mrcp_quant" / "optimized_layers" / "softmax"


In [ ]:
# Build/install after editing CUDA sources. Rerun this cell whenever softmax_cuda*.{cpp,cu}
# or common/tr_math.cuh changes, then restart/reload the notebook kernel if imports stay stale.
import shutil

softmax_dir = Path(SOFTMAX_EXT_DIR)
shutil.rmtree(softmax_dir / "build", ignore_errors=True)

for p in softmax_dir.glob("_trptq_softmax_dp4a*.so"):
    p.unlink()

!{sys.executable} -m pip install -v --no-build-isolation --no-cache-dir -e "{SOFTMAX_EXT_DIR}"

import _trptq_softmax_dp4a
print(_trptq_softmax_dp4a.__file__)

In [ ]:
import importlib

import trptq_softmax_dp4a
import _trptq_softmax_dp4a

trptq_softmax_dp4a = importlib.reload(trptq_softmax_dp4a)
print("Loaded:", trptq_softmax_dp4a.__file__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name())

for name in ["recip_u8", "recip_u16", "recip_q16_i32"]:
    if not hasattr(_trptq_softmax_dp4a, name):
        raise RuntimeError(
            f"The loaded _trptq_softmax_dp4a binary is stale and does not export {name}. "
            "Run the install cell again, then restart this notebook kernel/runtime before continuing."
        )
print("reciprocal exports: OK")


In [ ]:
def baseline_recip_u8(x):
    q = torch.round((1.0 / x.float()) * RECIP_UINT8_SCALE)
    return q.clamp(0, 255).to(torch.uint8)


def baseline_recip_u16(x):
    q = torch.round((1.0 / x.float()) * RECIP_UINT16_SCALE)
    return q.clamp(0, 65535).to(torch.uint16)


def baseline_recip_q16_i32(x):
    q = torch.round((1.0 / x.float()) * RECIP_Q16_SCALE)
    return q.clamp(0, RECIP_Q16_SCALE).to(torch.int32)


def sampled_int32_denominators(device):
    fixed = torch.tensor(
        [
            1, 2, 3, 4, 8, 15, 16, 17, 31, 32, 33,
            127, 128, 129, 255, 256, 257,
            1023, 1024, 1025, 65535, 65536, 65537,
            (1 << 20) - 1, 1 << 20, (1 << 20) + 1,
            (1 << 24) - 1, 1 << 24, (1 << 24) + 1,
            (1 << 30) - 1, 1 << 30, (1 << 30) + 1,
            INT32_MAX - 2, INT32_MAX - 1, INT32_MAX,
        ],
        device=device,
        dtype=torch.int64,
    )
    powers = 2 ** torch.arange(0, 31, device=device, dtype=torch.int64)
    log_grid = torch.logspace(0, math.log10(INT32_MAX), steps=4096, device=device)
    log_grid = log_grid.round().clamp(1, INT32_MAX).to(torch.int64)
    return torch.unique(torch.cat([fixed, powers, log_grid])).to(torch.int32)


def summarize(name, approx, baseline, scale):
    approx_cpu = approx.detach().cpu().to(torch.int64)
    baseline_cpu = baseline.detach().cpu().to(torch.int64)
    diff_q = approx_cpu - baseline_cpu
    diff_f = diff_q.double() / float(scale)
    return {
        "name": name,
        "max_abs_q": int(diff_q.abs().max().item()),
        "mean_abs_q": diff_q.abs().double().mean().item(),
        "max_abs_float": diff_f.abs().max().item(),
        "mean_abs_float": diff_f.abs().mean().item(),
        "rmse_float": diff_f.square().mean().sqrt().item(),
    }


def print_summary(title, rows):
    print(title)
    print(f"{'variant':<22} {'max_q':>8} {'mean_q':>10} {'max_f':>12} {'mean_f':>12} {'rmse_f':>12}")
    print("-" * 82)
    for row in rows:
        print(
            f"{row['name']:<22} {row['max_abs_q']:8d} {row['mean_abs_q']:10.4f} "
            f"{row['max_abs_float']:12.4e} {row['mean_abs_float']:12.4e} {row['rmse_float']:12.4e}"
        )


def selected_indices(x_cpu):
    selected = [
        1, 2, 3, 4, 8, 16, 32, 128, 255, 256, 257,
        1024, 65535, 65536, 65537, 1 << 20, 1 << 24, 1 << 30, INT32_MAX,
    ]
    value_to_index = {int(v): i for i, v in enumerate(x_cpu.tolist())}
    return [value_to_index[v] for v in selected if v in value_to_index]


def print_detail_table(title, x_cpu, approx, baseline, scale):
    approx_cpu = approx.detach().cpu().to(torch.int64)
    baseline_cpu = baseline.detach().cpu().to(torch.int64)
    err = approx_cpu - baseline_cpu
    print(title)
    print(f"{'x':>12} {'base_q':>8} {'cuda_q':>8} {'err_q':>8} {'base_f':>11} {'cuda_f':>11}")
    print("-" * 66)
    for i in selected_indices(x_cpu):
        base_q = int(baseline_cpu[i])
        cuda_q = int(approx_cpu[i])
        print(
            f"{int(x_cpu[i]):12d} {base_q:8d} {cuda_q:8d} {int(err[i]):8d} "
            f"{base_q / scale:11.6f} {cuda_q / scale:11.6f}"
        )


def print_worst_cases(title, x_cpu, approx, baseline, scale, k=12):
    approx_cpu = approx.detach().cpu().to(torch.int64)
    baseline_cpu = baseline.detach().cpu().to(torch.int64)
    err = approx_cpu - baseline_cpu
    topk = torch.topk(err.abs(), k=min(k, err.numel())).indices
    print(title)
    print(f"{'x':>12} {'base_q':>8} {'cuda_q':>8} {'err_q':>8} {'base_f':>11} {'cuda_f':>11}")
    print("-" * 66)
    for i in topk.tolist():
        base_q = int(baseline_cpu[i])
        cuda_q = int(approx_cpu[i])
        print(
            f"{int(x_cpu[i]):12d} {base_q:8d} {cuda_q:8d} {int(err[i]):8d} "
            f"{base_q / scale:11.6f} {cuda_q / scale:11.6f}"
        )


In [ ]:
assert torch.cuda.is_available(), "This notebook uses the CUDA extension; select a CUDA runtime first."
device = torch.device("cuda")

x_recip = sampled_int32_denominators(device)
baseline_u8 = baseline_recip_u8(x_recip)
baseline_u16 = baseline_recip_u16(x_recip)
cuda_u8 = trptq_softmax_dp4a.reciprocal_u8(x_recip)
cuda_u16 = trptq_softmax_dp4a.reciprocal_u16(x_recip)

one_idx = (x_recip == 1).nonzero(as_tuple=True)[0][0]
assert int(cuda_u8[one_idx]) == 255
assert int(cuda_u16[one_idx]) == 65535

x_q16 = torch.arange(1, (1 << 16) + 1, device=device, dtype=torch.int32)
baseline_q16 = baseline_recip_q16_i32(x_q16)
cuda_q16 = trptq_softmax_dp4a.reciprocal_q16_i32(x_q16)

print_summary(
    "Raw int32 reciprocal outputs",
    [
        summarize("reciprocal_u8", cuda_u8, baseline_u8, RECIP_UINT8_SCALE),
        summarize("reciprocal_u16", cuda_u16, baseline_u16, RECIP_UINT16_SCALE),
    ],
)
print()
print_summary(
    "Legacy int32 Q16 reciprocal output",
    [summarize("reciprocal_q16_i32", cuda_q16, baseline_q16, RECIP_Q16_SCALE)],
)


In [ ]:
x_recip_cpu = x_recip.cpu()
print_detail_table("uint8 reciprocal: raw int32 denominator -> Q8", x_recip_cpu, cuda_u8, baseline_u8, RECIP_UINT8_SCALE)
print()
print_detail_table("uint16 reciprocal: raw int32 denominator -> Q16", x_recip_cpu, cuda_u16, baseline_u16, RECIP_UINT16_SCALE)
print()
print_detail_table("legacy int32 Q16 reciprocal: x in [1, 65536]", x_q16.cpu(), cuda_q16, baseline_q16, RECIP_Q16_SCALE)


In [ ]:
print_worst_cases("Worst uint8 reciprocal cases", x_recip_cpu, cuda_u8, baseline_u8, RECIP_UINT8_SCALE)
print()
print_worst_cases("Worst uint16 reciprocal cases", x_recip_cpu, cuda_u16, baseline_u16, RECIP_UINT16_SCALE)
print()
print_worst_cases("Worst legacy int32 Q16 reciprocal cases", x_q16.cpu(), cuda_q16, baseline_q16, RECIP_Q16_SCALE)


In [ ]:
# Compare reciprocal kernels against torch reciprocal + quantization on random inputs.
x_big = torch.randint(1, INT32_MAX, (1_000_000,), device=device, dtype=torch.int32)
x_big_q16 = torch.randint(1, (1 << 16) + 1, (1_000_000,), device=device, dtype=torch.int32)


def benchmark_cuda(fn, warmup=20, repeats=200):
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(repeats):
        fn()
    end.record()
    torch.cuda.synchronize()
    return start.elapsed_time(end) / repeats


recip_u8_ms = benchmark_cuda(lambda: trptq_softmax_dp4a.reciprocal_u8(x_big))
torch_u8_ms = benchmark_cuda(lambda: baseline_recip_u8(x_big))

recip_u16_ms = benchmark_cuda(lambda: trptq_softmax_dp4a.reciprocal_u16(x_big))
torch_u16_ms = benchmark_cuda(lambda: baseline_recip_u16(x_big))

recip_q16_ms = benchmark_cuda(lambda: trptq_softmax_dp4a.reciprocal_q16_i32(x_big_q16))
torch_q16_ms = benchmark_cuda(lambda: baseline_recip_q16_i32(x_big_q16))

print(f"{'kernel':<20} {'ms':>10} {'speedup vs torch':>18}")
print("-" * 51)
print(f"{'reciprocal_u8':<20} {recip_u8_ms:10.4f} {torch_u8_ms / recip_u8_ms:18.2f}x")
print(f"{'torch_u8':<20} {torch_u8_ms:10.4f} {'1.00':>18}x")
print(f"{'reciprocal_u16':<20} {recip_u16_ms:10.4f} {torch_u16_ms / recip_u16_ms:18.2f}x")
print(f"{'torch_u16':<20} {torch_u16_ms:10.4f} {'1.00':>18}x")
print(f"{'reciprocal_q16_i32':<20} {recip_q16_ms:10.4f} {torch_q16_ms / recip_q16_ms:18.2f}x")
print(f"{'torch_q16_i32':<20} {torch_q16_ms:10.4f} {'1.00':>18}x")


In [ ]:
# e ^ (x-x_max) / (e ^ (x-x_max) + e ^ -x_max) = exp(x-x_max - ln(1 + exp(-abs(x_max))))

#  x_max -> -abs() -> exp() -> +1 -> ln() -> res_tmp

# -> x-x_max - res_tmp -> exp() 